<a href="https://colab.research.google.com/github/hagopjay/x3d.lab/blob/main/notebooks/05_GPU_Hog.00.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/hagopjay/x3d.lab/blob/main/notebooks/05_GPU_Hog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# x3d.lab · 05 GPU Hog

**One corpus, every engine, and findings sorted by the project that has to fix them.** About 230 scenes (one per X3D node from x3d.loom's bridge fixtures, the seven node types new in X3D 4.1, and the lab's own scenes), each as X3D 4.0 and 4.1, go through:

| Engine | What it answers |
|---|---|
| **x3d_mcp** (the fork) and **lxml** with the version's own XSD | valid? which error kinds? do two schema engines agree? |
| **X_ITE** in Chrome | loads? console errors? draws anything? (on the GPU if the machine allows, see step 3) |
| **Blender** + x3d-blender-io | imports what? round-trips to 4.0 and 4.1? is the exported file valid X3D? which node types come back? |
| **Cycles** on the GPU | draws the imported scene? |
| **x3d.loom** Babylon bridge | its reader + Babylon against X_ITE, node by node |

Then the **heavy** 02 Lab stress test on the GPU (after a warm-up, so first-render kernel loading is measured apart) and Chrome traces of growing meshes. Every finding names its project: **x3d_mcp, X_ITE, x3d-blender-io, x3d.loom, Copilot**, with the scene that shows it, and each project gets an issue draft.

**Runtime:** *Runtime › Change runtime type* › **A100** or **G4** (L4 works, slower). On a CPU runtime the stages still run, but the render and trace numbers are CPU numbers. **Time:** about 30–60 minutes with every stage on an A100.

In [1]:
!nvidia-smi

Wed Nov 11 00:33:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1 · Start the lab

In [3]:
#@title 1 · Start x3d_mcp and the dashboard { display-mode: "form" }
LOG_LEVEL = "WARN"  #@param ["DEBUG", "INFO", "WARN", "ERROR"]
LAB_REPO = "hagopjay/x3d.lab"
LAB_REF = "main"

import importlib.util, os, subprocess, sys
if not os.environ.get("X3DLAB_DEV") and (importlib.util.find_spec("x3dlab") is None or os.environ.get("X3DLAB_REINSTALL")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "x3dlab[notebook,tunnels,research] @ git+https://github.com/%s@%s" % (LAB_REPO, LAB_REF)], check=True)
import x3dlab
from x3dlab import hog

lab = x3dlab.up(tunnel="none", log_level=LOG_LEVEL)
print(lab.runtime_info["line"])
if "CPU" in lab.runtime_info["line"].split("|")[-1] and "NVIDIA" not in lab.runtime_info["line"]:
    print("⚠️  No GPU on this runtime: everything runs, but render and trace numbers will be CPU numbers.")

local: gateway answers,ok,200
local: dashboard key works,ok,200 mcp=up
local: wrong key refused,ok,401
local: live event stream,ok,first bytes arrived
local: MCP refuses no token,ok,401
local: MCP tools/list with token,ok,48 tools through the proxy


Python 3.13.16 | 12 CPUs | 167 GB RAM | NVIDIA A100-SXM4-80GB 80 GB | Colab


## 2 · Blender and the GPU

The same setup as 02 Lab. The first GPU render in a fresh runtime loads Cycles' kernels (minutes on an A100); the stress stage does a warm-up render first and reports that time separately.

In [4]:
#@title 2 · Set up Blender { display-mode: "form" }
BLENDER_SOURCE = "auto"  #@param ["auto", "wheel", "tarball"]
bl = lab.setup_blender(BLENDER_SOURCE)
print("Blender %s · best Cycles backend: %s · backends: %s" % (bl.version, bl.best_backend, bl.info.get("backends")))

Blender 5.2.3 LTS · best Cycles backend: OPTIX · backends: {'OPTIX': ['NVIDIA A100-SXM4-80GB'], 'CUDA': ['NVIDIA A100-SXM4-80GB'], 'HIP': [], 'ONEAPI': []}


## 3 · Probe the machine

What the engines will really get. Compute (CUDA, OptiX ray tracing) can work while the **graphics** half of NVIDIA's driver is missing from the container; then Chrome falls back to SwiftShader (WebGL on the CPU) and the OptiX *denoiser* cannot start. `ENABLE_GRAPHICS` installs that half (`libnvidia-gl` at exactly your driver's version, plus `vulkan-tools`), the fix from Google's headless-Chrome WebGPU notebook; then this cell says what the engines really get, from the files that are actually there.

In [10]:
#@title 3 · Probe { display-mode: "form" }
ENABLE_GRAPHICS = False  #@param {type:"boolean"}
if ENABLE_GRAPHICS:                     # the graphics half of the NVIDIA driver: Vulkan for Chrome, nvoptix.bin for the denoiser
    from x3dlab import graphics
    ge = graphics.enable()
    print("graphics:", ge.get("installed") or ge.get("skipped"), "| Vulkan devices:", (ge.get("after") or ge["before"])["vulkan_devices"] or "none")
probe = hog.probe(lab)
g = probe["graphics"]
print("GPU:", g["gpu"] or "none", "| NVIDIA_DRIVER_CAPABILITIES:", g["driver_capabilities"])
print("NVIDIA libraries:", ", ".join("%s %s" % (k, "✓" if v else "✗") for k, v in g["libs"].items()))
print("Vulkan ICDs:", g["vulkan_icds"] or "none", "| nvoptix.bin:", g["nvoptix_bin"] or "missing")
for v in g["verdicts"]:
    print("→", v)
for mode, info in probe["chrome"].items():
    print("Chrome %-11s WebGL: %s" % (mode, info.get("webgl") or info.get("error")))

GPU: NVIDIA A100-SXM4-80GB, 580.82.07, 81920 MiB, 8.0 | NVIDIA_DRIVER_CAPABILITIES: compute,utility
NVIDIA libraries: libGLX_nvidia ✓, libEGL_nvidia ✓, libnvidia-glcore ✓, libnvidia-eglcore ✓, libnvidia-rtcore ✓, libnvoptix ✓, libnvidia-glvkspirv ✓, libcuda ✓, libnvidia-ml ✗
Vulkan ICDs: ['/usr/share/vulkan/icd.d/nvidia_icd.json'] | nvoptix.bin: ['/usr/share/nvidia/nvoptix.bin']
→ Chrome can use the GPU
→ OptiX denoiser should work
Chrome swiftshader WebGL: ANGLE (Google, Vulkan 1.3.0 (SwiftShader Device (Subzero) (0x0000C0DE)), SwiftShader driver)
Chrome gpu         WebGL: ANGLE (NVIDIA, Vulkan 1.4.312 (NVIDIA NVIDIA A100-SXM4-80GB (0x000020B2)), NVIDIA)


In [11]:
#@title Why is Chrome still on SwiftShader?
from x3dlab import aio, browser as B
from playwright.async_api import async_playwright
FLAGS = ["--no-sandbox", "--use-angle=vulkan", "--enable-features=Vulkan", "--disable-vulkan-surface",
         "--enable-unsafe-webgpu", "--ignore-gpu-blocklist"]
async def gpu_page(flags):
    async with async_playwright() as pw:
        br = await pw.chromium.launch(headless=True, args=flags, executable_path=B.chromium_path())
        p = await br.new_page(); await p.goto("chrome://gpu"); await p.wait_for_timeout(2000)
        t = await p.inner_text("body"); await br.close(); return t
txt = aio.run(gpu_page(FLAGS))
keys = ("WebGL:", "WebGPU:", "Vulkan:", "GL_RENDERER", "GPU0", "Disabled Features", "ANGLE", "vulkan", "Problems", "Error")
seen = set()
for line in txt.splitlines():
    if any(k in line for k in keys) and line not in seen:
        seen.add(line); print(line.strip()[:220])
print("\nChromium:", B.chromium_path())


Chromium: /root/.cache/ms-playwright/chromium-1254/chrome-linux64/chrome


In [6]:
#@title 3b · Why the graphics install failed, and a pinned retry
print("first attempt said:\n", (ge.get("error") or "(no error text)")[-1500:])
import subprocess, re
V, PKG = "580.82.07-0ubuntu1", "libnvidia-gl-580"
deps = subprocess.run(["apt-cache", "depends", f"{PKG}={V}"], capture_output=True, text=True).stdout
siblings = sorted(set(re.findall(r"Depends: (libnvidia-[a-z0-9-]+-580)\b", deps)))
print("sibling packages to pin:", siblings)
cmd = ["apt-get", "install", "-y", "--no-install-recommends", f"{PKG}={V}", "vulkan-tools"] + [f"{s}={V}" for s in siblings]
r = subprocess.run(cmd, capture_output=True, text=True)
print("exit", r.returncode, "\n", (r.stdout + r.stderr)[-2500:])
print(subprocess.run("vulkaninfo --summary 2>&1 | grep -E 'deviceName|driverName'", shell=True, capture_output=True, text=True).stdout)

first attempt said:
 E: Unable to correct problems, you have held broken packages.

sibling packages to pin: ['libnvidia-common-580', 'libnvidia-compute-580', 'libnvidia-gpucomp-580']
exit 100 
 Reading package lists...
Building dependency tree...
Reading state information...
Some packages could not be installed. This may mean that you have
requested an impossible situation or if you are using the unstable
distribution that some required packages have not yet been created
or been moved out of Incoming.
The following information may help to resolve the situation:

The following packages have unmet dependencies:
 libnvidia-compute-580 : Depends: libnvidia-decode-580 (>= 580.82.07) but it is not going to be installed
                         Depends: libnvidia-cfg1-580 (= 580.82.07-0ubuntu1) but 580.196.07-1ubuntu1 is to be installed
E: Unable to correct problems, you have held broken packages.




In [7]:
#@title 3c · Unpack the graphics libraries directly (no apt dependency solving, CUDA untouched)
import subprocess, os, glob
V = "580.82.07-0ubuntu1"
os.makedirs("/tmp/nvgl", exist_ok=True)
pk = ["libnvidia-gl-580", "libnvidia-gpucomp-580", "libnvidia-common-580"]
r = subprocess.run(["apt-get", "download"] + [f"{p}={V}" for p in pk], cwd="/tmp/nvgl", capture_output=True, text=True)
print("download:", r.returncode, r.stderr[-400:])
for deb in sorted(glob.glob("/tmp/nvgl/*.deb")):
    subprocess.run(["dpkg", "-x", deb, "/"], check=True)      # unpack the files only; apt's records stay unchanged
    print("unpacked", os.path.basename(deb))
subprocess.run(["ldconfig"])
subprocess.run(["apt-get", "install", "-y", "-qq", "vulkan-tools"], capture_output=True)
print(subprocess.run("ls /usr/share/vulkan/icd.d; ls /usr/share/nvidia 2>/dev/null; vulkaninfo --summary 2>&1 | grep -E 'deviceName|driverName|ERROR'",
                     shell=True, capture_output=True, text=True).stdout)

download: 0 W: Download is performed unsandboxed as root as file '/tmp/nvgl/libnvidia-common-580_580.82.07-0ubuntu1_all.deb' couldn't be accessed by user '_apt'. - pkgAcquire::Run (13: Permission denied)

unpacked libnvidia-common-580_580.82.07-0ubuntu1_all.deb
unpacked libnvidia-gl-580_580.82.07-0ubuntu1_amd64.deb
unpacked libnvidia-gpucomp-580_580.82.07-0ubuntu1_amd64.deb
nvidia_icd.json
nvidia-application-profiles-580.82.07-key-documentation
nvidia-application-profiles-580.82.07-rc
nvoptix.bin
	deviceName         = NVIDIA A100-SXM4-80GB
	driverName         = NVIDIA



## 4 · Choose the stages

Everything is on by default. Turn a stage off to save time; the report covers whatever ran. `MAX_NODES` caps the corpus for a quick look (0 = all of it).

In [8]:
#@title 4 · Settings { display-mode: "form" }
VALIDATE_AND_RENDER = True  #@param {type:"boolean"}
LOOM_BABYLON = True  #@param {type:"boolean"}
STRESS = True  #@param {type:"boolean"}
STRESS_PRESET = "standard"  #@param ["standard", "heavy"]
TRACES = True  #@param {type:"boolean"}
X3D_41 = True  #@param {type:"boolean"}
CONSORTIUM = False  #@param {type:"boolean"}
CHROME_GPU = True  #@param {type:"boolean"}
MAX_NODES = 0  #@param {type:"integer"}
TRACE_SIZES = "20000, 200000, 1000000"  #@param {type:"string"}
NAME = ""  #@param {type:"string"}

stages = ["probe", "corpus"] + (["matrix"] if VALIDATE_AND_RENDER else []) + (["loom"] if LOOM_BABYLON else []) + \
         (["stress"] if STRESS else []) + (["traces"] if TRACES else []) + ["findings"]
cfg = hog.HogConfig(stages=stages, versions=["4.0", "4.1"] if X3D_41 else ["4.0"], consortium=CONSORTIUM, chrome_gpu=CHROME_GPU,
                    max_items=MAX_NODES, stress_preset=STRESS_PRESET, name=NAME,
                    trace_sizes=[int(x) for x in TRACE_SIZES.replace(" ", "").split(",") if x])
if CONSORTIUM and lab.upstream_client is None:
    lab.start_upstream(8001)
print("stages:", ", ".join(stages))

stages: probe, corpus, matrix, loom, stress, traces, findings


## 5 · Run the hog

Progress streams to the panel under cell 1 and to the dashboard. Each stage writes its results as it finishes, so a stopped run still leaves a report.

In [12]:
#@title 5 · Run it
run = lab.hog(cfg)
hog.show(run, lab)

**Reading it.**

- **Findings** come first, grouped by project. Each says how many scenes show it and quotes the evidence; the scene files are in the run folder under `scenes/`.
- The **matrix** has one row per scene: ✓/✗ per engine, plus how much of the frame X_ITE and Cycles covered (0 = drew nothing; event and sound nodes draw nothing by design).
- **export valid** is the new Blender check: is the file Blender wrote valid X3D of the version it claims? A ✗ there is a bug report for x3d-blender-io with the exact file attached.
- The **probe** lines say whether Chrome and the OptiX denoiser could use the GPU. If not, X_ITE and trace timings are CPU timings, and the stress test's denoiser falls back to OpenImageDenoise.

## 6 · Issue drafts

One Markdown file per project in `issues/`, ready to paste into a GitHub issue (review it first: a finding is a measurement, and some will be spec questions rather than bugs).

In [13]:
#@title 6 · Show the issue drafts { display-mode: "form" }
from x3dlab import ui
for f in sorted((run.dir / "issues").glob("*.md")):
    ui.show("<details><summary style='font:14px system-ui'><b>%s</b></summary><pre style='white-space:pre-wrap;font-size:12px'>%s</pre></details>"
            % (ui.esc(f.name), ui.esc(f.read_text())))
print("run folder:", run.dir)

run folder: /content/x3dlab/out/hog_7506d7a4


In [1]:
#@title A100 job 1 · Cycles scaling, GPU only (~10 min)
import threading
from x3dlab import stress, labui
sc = stress.RunConfig.preset_for("heavy", seed=7)
for t in ("validate", "roundtrip", "drift", "load"):
    sc.tiers[t]["on"] = False                       # CPU work: not on the A100's clock
sc.tiers["render"].update(
    compare_cpu=False, denoise=True, adaptive=False, scene_size=200,
    samples=[64, 256, 1024, 4096],                  # adaptive off: time should now grow with samples
    resolutions=[[1920, 1080], [3840, 2160], [7680, 4320]],   # 2, 8, 33 megapixels
    geometry=[200, 2000, 10000])                    # beyond 10k, Blender's import (CPU) dominates
lab.blender.gpu = threading.Semaphore(2)
sc.name = "a100-render-scaling"
run = lab.stress(sc)
labui.show_run(run, lab)

ModuleNotFoundError: No module named 'x3dlab'

## 7 · Keep everything

Zips the run folder (report, cards, findings, issue drafts, every scene and Blender export, traces) for download, then stops the lab. Save the notebook to GitHub too (*File › Save a copy in GitHub*); no key or token is in it.

In [14]:
#@title 7 · Zip the run and stop { display-mode: "form" }
import shutil
z = shutil.make_archive(str(run.dir), "zip", run.dir)
print("zip:", z, "(%.1f MB)" % (os.path.getsize(z) / 1e6))
try:
    from google.colab import files
    files.download(z)
except Exception:
    pass
lab.down()

zip: /content/x3dlab/out/hog_7506d7a4.zip (0.0 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>